# Deploy an OpenVINO Policy on an SO101 Robot

This notebook loads an exported robot policy with the OpenVINO Physical AI runtime, checks that its inputs and outputs match an SO101 deployment, and optionally runs it on real hardware.

Real robot execution is disabled by default. Review the model, calibration, camera names, workspace, and emergency-stop procedure before enabling it.

## 1) Install OpenVINO Physical AI

The runtime APIs used below are newer than the latest PyPI release. Install the current package directly from the Physical AI repository with the SO101 and shared-camera transport extras.

The general notebook environment should already be installed as described in [the tutorials README](README.md).

In [ ]:
%pip install -q "physicalai[so101,transport] @ git+https://github.com/openvinotoolkit/physicalai.git@main"

## 2) Load an OpenVINO Policy

The default path downloads the official `OpenVINO/pi05-libero-fp16-ov` package from Hugging Face with `InferenceModel.from_pretrained`. It lets every developer validate model download and OpenVINO loading without preparing a local export.

Use one of two loading paths:

1. **Hugging Face (default):** keep the official model ID for a loading test, or replace it with your own published OpenVINO policy package.
2. **Local OpenVINO package:** point to an extracted policy package on disk. For real robot deployment, obtain this package using one of these workflows:
   - **Physical AI Studio export (recommended):** train or import the policy in a current Studio version, export the OpenVINO backend, download the export, and extract it locally. This workflow preserves the manifest and runtime metadata expected by the current Physical AI APIs.
   - **External training and export:** train on another platform, such as a CUDA system, then export a complete OpenVINO policy package yourself. Keep the model IR, manifest, preprocessing/postprocessing assets, normalization statistics, and feature schema together. A standalone `.xml`/`.bin` pair may not contain enough deployment metadata.

The official Pi0.5 checkpoint is trained for the LIBERO benchmark. Its 8-dimensional state and 7-dimensional action interface are not compatible with a six-motor SO101, so keep `RUN_ON_ROBOT = False` when using this default model.

See the [Physical AI Studio documentation](https://github.com/open-edge-platform/physical-ai-studio/blob/main/application/README.md) for its model and hardware workflows.

### Action required: Select and configure a model source

Keep `MODEL_SOURCE = "huggingface"` for the default loading validation, or set it to `"local"` for an extracted Studio export or an externally exported package.

- For `"huggingface"`, keep the official model ID or set `HUGGINGFACE_MODEL_ID` to your published policy package.
- For `"local"`, set `LOCAL_MODEL_PATH` to the root of the extracted OpenVINO policy package.
- Set `TASK` to the instruction used by a language-conditioned policy, or `None` for a policy that does not use language.
- Current Studio exports should declare feature metadata. For an older or external package that does not, set `FALLBACK_CAMERA_NAMES` to the exact camera feature names used during data collection and training.

Keep `RUN_ON_ROBOT = False` until the compatibility section confirms that the policy is intended for this SO101 and its six-dimensional state and action interfaces.

In [ ]:
from pathlib import Path

MODEL_SOURCE = "huggingface"

LOCAL_MODEL_PATH = Path("physicalai_assets/models/my-so101-openvino-export")

HUGGINGFACE_MODEL_ID = "OpenVINO/pi05-libero-fp16-ov"
HUGGINGFACE_REVISION = None
HUGGINGFACE_CACHE_DIR = Path("physicalai_assets/models")

DEVICE = "AUTO"
TASK = "pick up the box"
FALLBACK_CAMERA_NAMES = ["image", "image2"]
FPS = 30.0
DURATION_S = 60.0
RUN_ON_ROBOT = False

In [ ]:
from physicalai.inference import InferenceModel

if MODEL_SOURCE == "local":
    model_path = LOCAL_MODEL_PATH.expanduser().resolve()
    if not model_path.is_dir():
        raise FileNotFoundError(
            f"Local policy directory not found: {model_path}. "
            "Update LOCAL_MODEL_PATH or select MODEL_SOURCE='huggingface'."
        )
    policy = InferenceModel(
        model_path,
        backend="openvino",
        device=DEVICE,
    )
    print(f"Loaded local policy: {model_path}")
elif MODEL_SOURCE == "huggingface":
    policy = InferenceModel.from_pretrained(
        HUGGINGFACE_MODEL_ID,
        revision=HUGGINGFACE_REVISION,
        cache_dir=HUGGINGFACE_CACHE_DIR,
        backend="openvino",
        device=DEVICE,
    )
    print(f"Loaded Hugging Face policy: {HUGGINGFACE_MODEL_ID}")
else:
    raise ValueError("MODEL_SOURCE must be 'local' or 'huggingface'.")

print(policy)
if policy.input_features:
    print("Input features:")
    for feature in policy.input_features:
        print(f"  {feature.name}: {feature.ftype}, shape={feature.shape}, dtype={feature.dtype}")
else:
    print("The package does not declare input feature metadata; configure the documented fallbacks before deployment.")

if policy.output_features:
    print("Output features:")
    for feature in policy.output_features:
        print(f"  {feature.name}: {feature.ftype}, shape={feature.shape}, dtype={feature.dtype}")
else:
    print("The package does not declare output feature metadata; confirm its SO101 action schema before deployment.")

## 3) Configure SO101 Hardware

Skip this section when `RUN_ON_ROBOT` is `False`.

For a real deployment, keep the model, calibration, and camera configuration from the same data-collection workflow:

1. **Studio-assisted workflow (recommended):** download the OpenVINO model export and SO101 calibration JSON from a current Physical AI Studio setup. During camera selection, use the exact camera names, resolution, and FPS from the Studio environment used to collect the training dataset.
2. **External/manual workflow:** provide a compatible OpenVINO policy package and SO101 calibration JSON produced outside current Studio, then configure the physical cameras explicitly. The feature names and ordering must still match the training dataset.

### Linux device permissions

The notebook user needs access to the robot serial port, camera devices, and, when applicable, the OpenVINO GPU device. Check `groups` and device ownership before starting Jupyter. A typical one-time setup is:

```bash
sudo usermod -aG dialout,video,render "$USER"
```

Log out and back in after changing group membership. Do not run Jupyter as root.

### Action required: Select the calibration source

Choose one of two calibration paths:

1. **Studio download (recommended):** download the calibration JSON for this SO101 from a current Physical AI Studio robot configuration, place it in a local directory, and set `STUDIO_CALIBRATION_PATH`.
2. **Custom path:** for an older Studio setup, LeRobot, or another calibration workflow, set `CUSTOM_CALIBRATION_PATH` to its compatible SO101 calibration JSON.

Set `CALIBRATION_SOURCE` to `"studio_download"` or `"custom_path"`. The notebook does not assume a Studio cache location. The selected file must belong to this physical robot and contain calibration data for all six motors.

In [ ]:
SO101_PORT = "/dev/ttyACM0"

CALIBRATION_SOURCE = "studio_download"
STUDIO_CALIBRATION_PATH = Path("physicalai_assets/calibrations/so101_calibration.json")
CUSTOM_CALIBRATION_PATH = Path("/path/to/existing-so101-calibration.json")

if CALIBRATION_SOURCE == "studio_download":
    SO101_CALIBRATION = STUDIO_CALIBRATION_PATH.expanduser()
elif CALIBRATION_SOURCE == "custom_path":
    SO101_CALIBRATION = CUSTOM_CALIBRATION_PATH.expanduser()
else:
    raise ValueError("CALIBRATION_SOURCE must be 'studio_download' or 'custom_path'.")

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480
CAMERA_FPS = 30

CAMERA_SOURCE = "interactive"
MANUAL_CAMERA_CONFIGS = [
    {
        "name": "image",
        "camera_type": "uvc",
        "init_args": {"device": "/dev/v4l/by-id/usb-example-overhead-video-index0"},
    },
    {
        "name": "image2",
        "camera_type": "uvc",
        "init_args": {"device": "/dev/v4l/by-id/usb-example-arm-video-index0"},
    },
]

In [ ]:
from physicalai.robot import SO101

robot = None
if RUN_ON_ROBOT:
    if not SO101_CALIBRATION.is_file():
        raise FileNotFoundError(f"Calibration file not found: {SO101_CALIBRATION}")
    robot = SO101(
        port=SO101_PORT,
        calibration=SO101_CALIBRATION,
        role="follower",
    )
else:
    print("Robot setup skipped. Set RUN_ON_ROBOT = True after validating a compatible policy.")

## 4) Validate Deployment Compatibility

The runtime maps robot observations and named camera frames to the public feature schema in `policy.input_features`. The model state and action dimensions must match the robot joint count. Multi-camera names must match the suffixes of the model's visual feature names.

This validation uses only public APIs; it does not construct a private runtime input.

In [ ]:
def features_of_type(features, feature_type):
    return [feature for feature in features if feature.ftype == feature_type]


visual_features = features_of_type(policy.input_features, "VISUAL")
state_features = features_of_type(policy.input_features, "STATE")
action_features = features_of_type(policy.output_features, "ACTION")

declared_camera_names = [
    feature.name.removeprefix("images.")
    for feature in visual_features
]
if declared_camera_names:
    expected_camera_names = declared_camera_names
    camera_name_source = "model feature metadata"
else:
    expected_camera_names = list(FALLBACK_CAMERA_NAMES)
    camera_name_source = "FALLBACK_CAMERA_NAMES"
    if not expected_camera_names:
        raise ValueError(
            "The model has no visual feature metadata. "
            "Set FALLBACK_CAMERA_NAMES to the camera names used during training."
        )

if len(set(expected_camera_names)) != len(expected_camera_names):
    raise ValueError(f"Camera names must be unique: {expected_camera_names}")

state_dims = sorted({feature.shape[-1] for feature in state_features if feature.shape})
action_dims = sorted({feature.shape[-1] for feature in action_features if feature.shape})

print(f"Expected cameras ({camera_name_source}): {expected_camera_names}")
print(f"Expected state dimensions: {state_dims or 'none declared'}")
print(f"Expected action dimensions: {action_dims or 'none declared'}")

if RUN_ON_ROBOT:
    robot_dim = len(robot.joint_names)
    if state_dims and state_dims != [robot_dim]:
        raise ValueError(f"Model state dimensions {state_dims} do not match SO101 joint count {robot_dim}.")
    if action_dims and action_dims != [robot_dim]:
        raise ValueError(f"Model action dimensions {action_dims} do not match SO101 joint count {robot_dim}.")
    print(f"SO101 state/action dimensions match its {robot_dim} joints.")

### Action required: Configure cameras

Choose one of two camera paths:

1. **Interactive selection (recommended):** keep `CAMERA_SOURCE = "interactive"`. The public `select_cameras_interactive` API discovers connected cameras. Select one index at a time and assign the exact feature names printed by the compatibility cell. For a Studio-trained policy, use the names from the Studio environment used during data collection.
2. **Manual configuration:** set `CAMERA_SOURCE = "manual"` and update `MANUAL_CAMERA_CONFIGS`. This is useful for an external setup or unattended reruns. Prefer stable Linux device IDs such as `/dev/v4l/by-id/...-video-index0` instead of `/dev/videoN`.

Both paths create shared cameras through public Physical AI APIs. They are not connected yet; the `RobotRuntime` context manager connects and disconnects them together with the robot.

In [ ]:
from physicalai.capture import create_camera, select_cameras_interactive

cameras = {}
if RUN_ON_ROBOT:
    if CAMERA_SOURCE == "interactive":
        cameras = select_cameras_interactive(
            width=CAMERA_WIDTH,
            height=CAMERA_HEIGHT,
            fps=CAMERA_FPS,
        )
    elif CAMERA_SOURCE == "manual":
        for config in MANUAL_CAMERA_CONFIGS:
            name = config["name"]
            if name in cameras:
                raise ValueError(f"Duplicate manual camera name: {name}")
            cameras[name] = create_camera(
                config["camera_type"],
                shared=True,
                width=CAMERA_WIDTH,
                height=CAMERA_HEIGHT,
                fps=CAMERA_FPS,
                **config["init_args"],
            )
    else:
        raise ValueError("CAMERA_SOURCE must be 'interactive' or 'manual'.")

    if len(cameras) != len(expected_camera_names):
        raise ValueError(
            f"Selected {len(cameras)} camera(s), but {camera_name_source} "
            f"expects {len(expected_camera_names)}: {expected_camera_names}."
        )
    if set(cameras) != set(expected_camera_names):
        raise ValueError(
            f"Camera names {sorted(cameras)} do not match the expected names "
            f"{sorted(expected_camera_names)} from {camera_name_source}."
        )
else:
    print("Camera selection skipped.")

## 5) Run the Policy on the Robot

Clear the robot workspace and make the emergency stop accessible before running this cell. The `RobotRuntime` context manager owns hardware connection and cleanup, including cleanup after an exception or keyboard interrupt.

In [ ]:
from physicalai.runtime import PolicySource, RobotRuntime, SyncExecution

if RUN_ON_ROBOT:
    execution = SyncExecution(request_threshold=0.5)
    policy_source = PolicySource(
        model=policy,
        execution=execution,
        task=TASK,
    )
    runtime = RobotRuntime(
        robot=robot,
        action_source=policy_source,
        fps=FPS,
        cameras=cameras,
    )

    with runtime:
        steps = runtime.run(duration_s=DURATION_S)

    print(f"Completed steps: {steps}")
    print(f"Inference requests after warmup: {execution.inference_count}")
    print(f"Action queue holds: {policy_source.action_queue.total_holds}")
else:
    print("Deployment skipped. Set RUN_ON_ROBOT = True only with a compatible SO101 policy.")